# RAG Project

##  Document Loading and Text Extraction

In this step, we will:

- Load a PDF document.
- Extract the text.
- Display a sample of the extracted content.
- Check whether the extraction is clean and readable.

# Install Required Libraries

Run this cell only once when setting up the project.

In [78]:
import sys

packages = [
    "pypdf",
    "sentence-transformers",
    "chromadb",
    "transformers==4.55.4",
    "sentencepiece",
    "accelerate"
]

!{sys.executable} -m pip install -q {" ".join(packages)}

print(" All required libraries are installed successfully.")

 All required libraries are installed successfully.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [79]:
from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

import chromadb

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

import torch

## Load the PDF Document

In this step, we define the path of the PDF file and load it using `PdfReader`.

The PDF file must be located in the same project folder as the notebook.

In [80]:
pdf_path = "iso27001.pdf"

reader = PdfReader(pdf_path)

## Check the Number of Pages

Before extracting the text, we check how many pages are available in the PDF document.

In [81]:
number_of_pages = len(reader.pages)

print("Number of pages:", number_of_pages)

Number of pages: 26


## Extract Text from All Pages

In this step, we loop through all pages in the PDF and extract the text from each page.

The extracted text will be stored in one variable called `document_text`.

In [82]:
document_text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        document_text += page_text + "\n"

## Display a Sample of the Extracted Text

To verify that the text extraction worked correctly, we display a small sample from the beginning of the document.

In [83]:
sample_size = 1000

print(document_text[:sample_size])

Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
Third edition  
2022-10
Reference number 
ISO/IEC 27001:2022(E)
© ISO/IEC 2022
--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---
ii
ISO/IEC 27001:2022(E)
COPYRIGHT PROTECTED DOCUMENT
©  ISO/IEC 2022
All rights reserved. Unless otherwise specified, or required in the context of its implementation, no part of this publication may 
be reproduced or utilized otherwise in any form or by any means, electronic or mechanical, including photocopying, or posting on 
the internet or an intranet, without prior written permission. Permission can be requested from either ISO at the address below 
or ISO’s member body in the country of the requester.
ISO copyright office
CP 401 • Ch. de Blandonnet 8
CH-12

## Check the Extraction Quality

In this step, we perform a basic check to confirm that the extracted text is not empty and contains readable content.

In [84]:
text_length = len(document_text)
word_count = len(document_text.split())

print("Number of characters:", text_length)
print("Number of words:", word_count)

Number of characters: 57197
Number of words: 6748


## Final Extraction Output

This section displays the final extraction results, including:

- Document loading status
- Number of pages
- Total number of extracted characters
- A short sample from the extracted text

In [85]:
sample_size = 300

print("Document loaded successfully.")
print("Number of pages:", len(reader.pages))
print("Extracted characters:", len(document_text))
print("Sample:", document_text[:sample_size])

Document loaded successfully.
Number of pages: 26
Extracted characters: 57197
Sample: Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001


# Chunking

In this step, the extracted document text is divided into smaller chunks.

Chunking is necessary because embedding models work better with smaller sections of text rather than one long document.

Suggested starting values:

- Chunk Size: 500 characters
- Chunk Overlap: 50 characters

In [86]:
chunk_size = 500
chunk_overlap = 50

chunks = []

start = 0

while start < len(document_text):
    end = start + chunk_size

    chunk = document_text[start:end]

    chunks.append(chunk)

    start += chunk_size - chunk_overlap

## Check the Generated Chunks

This section displays:

- The total number of chunks
- The length of the first chunk
- A sample of the first chunk

In [87]:
print("Number of chunks:", len(chunks))
print("First chunk length:", len(chunks[0]))
print("\nFirst chunk sample:")
print(chunks[0])

Number of chunks: 128
First chunk length: 500

First chunk sample:
Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
Third edition  
2022-10
Reference number 
ISO/IEC 27001:2022(E)
© ISO/IEC 2022
--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---
ii
ISO/IEC 27001:2022(E)
COPYRIGHT PROTECTED DOCUMENT
©  ISO/IEC 2022



## Check Chunk Overlap

The last 50 characters of the first chunk should be the same as the first 50 characters of the second chunk.

In [88]:
print("End of Chunk 1:")
print(chunks[0][-50:])

print("\nStart of Chunk 2:")
print(chunks[1][:50])

End of Chunk 1:
2(E)
COPYRIGHT PROTECTED DOCUMENT
©  ISO/IEC 2022


Start of Chunk 2:
2(E)
COPYRIGHT PROTECTED DOCUMENT
©  ISO/IEC 2022



#  Create Embeddings

In this step, each text chunk is converted into a numerical vector called an embedding.

Embeddings represent the semantic meaning of text rather than only matching exact keywords.

This means that sentences with similar meanings should produce similar embedding vectors, even when they use different words.

Embeddings are not final answers. They are used later to perform similarity search and retrieve the most relevant chunks from the document.

## Load the Embedding Model

In this step, we load the embedding model that will convert each text chunk into a numerical vector.

We will use the **all-MiniLM-L6-v2** model because it is lightweight, fast, and commonly used in RAG applications.

In [89]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

## Generate Embeddings

In this step, each text chunk is converted into a numerical embedding vector.

These vectors capture the semantic meaning of the text and will later be used for similarity search.

In [90]:
chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

## Verify the Embeddings

This step checks whether the embeddings were created successfully.

We verify:

- Number of chunks
- Number of embeddings
- Embedding dimensions
- Sample values from the first embedding

In [91]:
print(f"Number of chunks: {len(chunks)}")
print(f"Number of embeddings: {len(chunk_embeddings)}")
print(f"Embedding shape: {chunk_embeddings.shape}")

print("\nFirst embedding (first 10 values):")
print(chunk_embeddings[0][:10])

Number of chunks: 128
Number of embeddings: 128
Embedding shape: (128, 384)

First embedding (first 10 values):
[-0.02654819  0.01417822 -0.0905083  -0.09985209  0.04653104  0.01956914
  0.08642682  0.03603141  0.05179968  0.00656525]


#  Store Chunks in a Vector Database

In this step, the text chunks and their embeddings are stored in a vector database.

A vector database is used to:

- Store chunk embeddings persistently.
- Search by semantic similarity when a query is received.
- Return the most relevant chunks.
- Pass the retrieved chunks to the Large Language Model (LLM).

## Create a Persistent ChromaDB Client

In this step, we create a persistent ChromaDB client.

The database will be stored locally inside the project folder, allowing the embeddings to be reused without recreating them every time.

In [92]:


chroma_client = chromadb.PersistentClient(
    path="chroma_db"
)

## Create a ChromaDB Collection

A collection is used to organize and store the document chunks and their embedding vectors.

This project will use a collection called `rag_documents`.

In [93]:
collection = chroma_client.get_or_create_collection(
    name="rag_documents"
)

## Prepare Data for Storage

Before storing the document in ChromaDB, we prepare:

- A unique ID for each chunk.
- Metadata describing each chunk.
- The original chunk text.
- The embedding vector.

In [94]:
chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]

chunk_metadata = [
    {
        "chunk_index": i,
        "source": "RAG_Sample_Knowledge_Base.pdf"
    }
    for i in range(len(chunks))
]

## Store Chunks and Embeddings

In this step, each document chunk is stored together with:

- A unique ID
- The original text
- The embedding vector
- Metadata

This information will later be used for semantic similarity search.

In [95]:
collection.add(
    ids=chunk_ids,
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunk_metadata
)

## Verify Stored Data

In this step, we verify that all chunks have been successfully stored in the vector database.

The output should match the total number of generated chunks.

In [96]:
stored_chunks = collection.count()

print("Stored chunks:", stored_chunks)

Stored chunks: 128


#  Retrieval

In this step, we ask a test question and retrieve the top 3 most relevant chunks from the vector database.

The retrieval process works by:

- Converting the question into an embedding.
- Comparing it with the stored chunk embeddings.
- Returning the most semantically relevant chunks.

In [97]:
question = "What is the main objective of the document?"

question_embedding = embedding_model.encode(
    question,
    convert_to_numpy=True
)

## Retrieve the Top 3 Relevant Chunks

The question embedding is compared with the embeddings stored in ChromaDB.

The database returns the top 3 chunks with the smallest distance values.

A smaller distance means higher semantic similarity.

In [98]:
top_k = 3

results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=top_k,
    include=["documents", "distances", "metadatas"]
)

## Display the Retrieved Chunks

This step displays:

- The user's question
- The top 3 retrieved chunks
- Their similarity (distance) scores

Lower distance values indicate higher semantic similarity.

In [99]:
print("Question:")
print(question)

print("\nTop Retrieved Chunks:\n")

for i in range(top_k):
    print(f"Result {i+1}")
    print(f"Distance Score: {results['distances'][0][i]:.4f}")
    print(results["documents"][0][i])
    print("-" * 80)

Question:
What is the main objective of the document?

Top Retrieved Chunks:

Result 1
Distance Score: 1.1286
n and its context
The organization shall determine external and internal issues that are relevant to its purpose and that 
affect its ability to achieve the intended outcome(s) of its information security management system.
NOTE Determining these issues refers to establishing the external and internal context of the organization 
considered in Clause 5.4.1 of ISO 31000:2018[5].
4.2  Understanding the needs and expectations of interested parties
The organization shall determine:
a) interested par
--------------------------------------------------------------------------------
Result 2
Distance Score: 1.1287
 document.
2  Normative references
The following documents are referred to in the text in such a way that some or all of their content 
constitutes requirements of this document. For dated references, only the edition cited applies. For 
undated references, the latest edition

# Generate the Answer

In this step, the retrieved chunks and the user's question will be sent to a language model.

The model must:

- Answer using only the retrieved context.
- State that the information is unavailable if the context does not contain the answer.
- Avoid inventing or inferring information outside the document.

In [100]:
retrieved_chunks = results["documents"][0]

retrieved_context = "\n\n".join(retrieved_chunks)

print("Retrieved Context:\n")
print(retrieved_context)

Retrieved Context:

n and its context
The organization shall determine external and internal issues that are relevant to its purpose and that 
affect its ability to achieve the intended outcome(s) of its information security management system.
NOTE Determining these issues refers to establishing the external and internal context of the organization 
considered in Clause 5.4.1 of ISO 31000:2018[5].
4.2  Understanding the needs and expectations of interested parties
The organization shall determine:
a) interested par

 document.
2  Normative references
The following documents are referred to in the text in such a way that some or all of their content 
constitutes requirements of this document. For dated references, only the edition cited applies. For 
undated references, the latest edition of the referenced document (including any amendments) applies.
ISO/IEC 27000, Information technology — Security techniques — Information security management 
systems — Overview and vocabulary
3	 	Terms

In [101]:
retrieved_context

'n and its context\nThe organization shall determine external and internal issues that are relevant to its purpose and that \naffect its ability to achieve the intended outcome(s) of its information security management system.\nNOTE Determining these issues refers to establishing the external and internal context of the organization \nconsidered in Clause 5.4.1 of ISO 31000:2018[5].\n4.2  Understanding the needs and expectations of interested parties\nThe organization shall determine:\na) interested par\n\n document.\n2  Normative references\nThe following documents are referred to in the text in such a way that some or all of their content \nconstitutes requirements of this document. For dated references, only the edition cited applies. For \nundated references, the latest edition of the referenced document (including any amendments) applies.\nISO/IEC 27000, Information technology — Security techniques — Information security management \nsystems — Overview and vocabulary\n3\t \tTerms\

## Create the Prompt

The prompt combines the retrieved context with the user's question.

The language model is instructed to:

- Answer only using the provided context.
- Do not invent information.
- If the answer is not available, respond with:
  "The information is not available in the provided document."

In [102]:
prompt = f"""
You are an AI assistant.

Use ONLY the context below to answer the user's question.

Rules:
1. Answer only from the provided context.
2. Do not invent or infer information.
3. If the answer is not available in the context, reply exactly:
"The information is not available in the provided document."

Context:
{retrieved_context}

Question:
{question}

Answer:
"""

## Generate the Final Answer

The prompt is sent to a Large Language Model (LLM).

The model generates a grounded answer using only the retrieved context.

If the answer is not found in the context, the model should respond:

"The information is not available in the provided document."

## Load the Language Model

The language model will generate the final answer based on the retrieved context and the user's question.

In [103]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

In [104]:
import transformers
import tokenizers
import sentencepiece

print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("sentencepiece:", sentencepiece.__version__)

transformers: 4.55.4
tokenizers: 0.21.4
sentencepiece: 0.2.2


In [105]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


## Load the FLAN-T5 Model

The tokenizer was loaded successfully.

In this step, we load the FLAN-T5 language model that will generate the final grounded answer.

In [106]:
from transformers import AutoModelForSeq2SeqLM

llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

print("Model loaded successfully!")

Model loaded successfully!


## Generate the Final Answer

The retrieved context and the user's question are combined into a prompt.

The language model generates an answer using only the retrieved context.

If the answer is not available, the model should respond that the information is not available in the provided document.

In [107]:
question = "What is the main objective of the document?"

results = collection.query(
    query_texts=[question],
    n_results=3
)

retrieved_chunks = results["documents"][0]

context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using only the provided context.

If the answer is not available in the context, say:
"The information is not available in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

In [108]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = llm_model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What is the main objective of the document?

Answer:
The organization shall determine external and internal issues that are relevant to its purpose and that affect its ability to achieve the intended outcome(s) of its information security management system



#  Test the RAG System

In this step, the RAG system is evaluated using different types of questions.

The evaluation includes:

- 5 questions with answers available in the document.
- 3 questions that require semantic understanding.
- 2 questions whose answers are not available in the document.

The system should answer only from the retrieved context and avoid hallucinating information.

In [109]:
test_questions = [

    # ==========================
    # Available Questions (5)
    # ==========================

    {
        "category": "Available",
        "question": "What does ISO/IEC 27001 specify?"
    },

    {
        "category": "Available",
        "question": "What is the title of Clause 4?"
    },

    {
        "category": "Available",
        "question": "What is the title of Clause 5?"
    },

    {
        "category": "Available",
        "question": "What is the title of Clause 10?"
    },

    {
        "category": "Available",
        "question": "What document contains the terms and definitions?"
    },

    # ==========================
    # Detailed Questions (3)
    # ==========================

    {
        "category": "Detailed",
        "question": "Summarize the purpose of the information security management system."
    },

    {
        "category": "Detailed",
        "question": "Explain the purpose of internal audits."
    },

    {
        "category": "Detailed",
        "question": "What should an organization do when a nonconformity occurs?"
    },

    # ==========================
    # Unavailable Questions (2)
    # ==========================

    {
        "category": "Unavailable",
        "question": "Who created ISO/IEC 27001?"
    },

    {
        "category": "Unavailable",
        "question": "What is the certification cost of ISO/IEC 27001?"
    }

]

## Run the Test Questions

Each question is processed through the complete RAG pipeline.

For every question, the system:

1. Converts the question into an embedding.
2. Retrieves the three most relevant chunks.
3. Sends the retrieved context to the language model.
4. Generates a grounded answer.

In [110]:
test_results = []

for index, item in enumerate(test_questions, start=1):
    question = item["question"]

    # Convert the question into an embedding
    question_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True
    )

    # Retrieve the top 3 relevant chunks
    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=3,
        include=["documents", "distances", "metadatas"]
    )

    retrieved_chunks = results["documents"][0]
    distances = results["distances"][0]

    context = "\n\n".join(retrieved_chunks)

    # Create a grounded prompt
    prompt = f"""
Answer the question using only the provided context.

Rules:
1. Use only information explicitly available in the context.
2. Do not use outside knowledge.
3. Do not invent or infer missing information.
4. If the answer is not clearly available, reply exactly:
"The information is not available in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    # Convert the prompt into model inputs
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Generate the grounded answer
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Save the test result
    test_results.append(
        {
            "test_number": index,
            "category": item["category"],
            "question": question,
            "answer": answer,
            "retrieved_chunks": retrieved_chunks,
            "distances": distances
        }
    )

    # Display the result
    print("=" * 100)
    print(f"Test {index}")
    print(f"Category: {item['category']}")
    print(f"Question: {question}")

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Chunk Distances:")
    for rank, distance in enumerate(distances, start=1):
        print(f"{rank}. {distance:.4f}")

    print()

Test 1
Category: Available
Question: What does ISO/IEC 27001 specify?

Answer:
the overview and the vocabulary of information security management systems

Retrieved Chunk Distances:
1. 0.5033
2. 0.5820
3. 0.6367

Test 2
Category: Available
Question: What is the title of Clause 4?

Answer:
The organization shall determine external and internal issues that are relevant to its purpose and that affect its ability to achieve the intended outcome(s) of its information security management system.

Retrieved Chunk Distances:
1. 1.3405
2. 1.3731
3. 1.3776

Test 3
Category: Available
Question: What is the title of Clause 5?

Answer:
The organization shall determine external and internal issues that are relevant to its purpose and that affect its ability to achieve the intended outcome(s) of its information security management system.

Retrieved Chunk Distances:
1. 1.3228
2. 1.3731
3. 1.3833

Test 4
Category: Available
Question: What is the title of Clause 10?

Answer:
The Organization shall dete

In [111]:
document_text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        document_text += page_text + "\n"

In [112]:
print("Document loaded successfully.")
print("Number of pages:", len(reader.pages))
print("Extracted characters:", len(document_text))
print("Sample:", document_text[:300])
print("Sample:", document_text[:300])

Document loaded successfully.
Number of pages: 26
Extracted characters: 57197
Sample: Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
Sample: Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèmes de management de la sécurité de l'information — 
Exigences
INTERNATIONAL 
STANDARD
ISO/IEC 
27001
